# 🧠 Day 7 · Generative AI Fundamentals
## Build Your Own GPT From Scratch: no API keys, no black boxes

Today we build **every part of a ChatGPT-style system ourselves**, and then wrap it in a real app.

```
📚 Data → 🔤 Tokenizer → 🧭 Embeddings → 👀 Transformer → 🏋️ Training → 🎲 Generation → 🎓 Instruction tuning → 🛡️ Safety → 📱 App
 Part 1      Part 2         Part 3          Part 4         Part 5-6       Part 7            Part 8                 Part 9-10    Part 11
```

| Real ChatGPT-class model | Our StoryGPT (today) |
|---|---|
| trillions of training tokens | ~1.5 million tokens (children's stories) |
| 100+ billion parameters | ~1 million parameters |
| weeks on thousands of GPUs | a few minutes on one free Colab GPU |
| **same ideas, same maths, same code structure** | ✅ |

**Before you start:** `File → Save a copy in Drive`, then `Runtime → Change runtime type → T4 GPU`.
Run cells **top to bottom** with `Shift + Enter`.

---
# Part 0 · Setup

In [ ]:
# Colab already has PyTorch, NumPy and Matplotlib. We only add two small libraries:
#   tiktoken -> the real GPT-4o tokenizer (runs locally, only to COMPARE with ours)
#   gradio   -> to build the web app at the end
%pip install -q tiktoken gradio

In [ ]:
import os, re, math, time, json, random, copy, html, urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import HTML, display

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
random.seed(1337)
print("PyTorch:", torch.__version__)
if device == "cuda":
    print("✅ GPU found:", torch.cuda.get_device_name(0), "- training will take ~2 minutes")
else:
    print("⚠️ No GPU. Runtime → Change runtime type → T4 GPU.  (CPU works too: we use a smaller model)")

---
# Part 1 · Data: what does an LLM learn from?

A **Large Language Model (LLM)** is trained on one simple task:

> **Given some text, predict the next token.**

That's the whole training objective of GPT. Everything else (stories, code, answers) *emerges* from doing this
extremely well on a huge amount of text.

We use **TinyStories**: 6,000 short stories in simple English (Eldan & Li, Microsoft Research, 2023, license CDLA-Sharing-1.0).
It's a public dataset, so no login or key is needed.

In [ ]:
RAW_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"
EOT = "<|endoftext|>"          # a special marker between documents

if not os.path.exists("TinyStories-valid.txt"):
    print("Downloading TinyStories (19 MB)...")
    urllib.request.urlretrieve(RAW_URL, "TinyStories-valid.txt")
raw = open("TinyStories-valid.txt", encoding="utf-8").read()

# ---- Data cleaning: real LLM teams spend a LOT of time here ----
REPLACEMENTS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00a0": " "}

def clean(story):
    for bad, good in REPLACEMENTS.items():
        story = story.replace(bad, good)
    return story.strip()

stories = []
for story in raw.split(EOT):
    story = clean(story)
    if len(story) < 50 or not story.isascii():   # quality filter
        continue
    stories.append(story)
    if len(stories) == 6000:
        break

text = "".join(s + "\n" + EOT + "\n" for s in stories)
open("stories.txt", "w", encoding="utf-8").write(text)
print(f"Stories: {len(stories):,}   Characters: {len(text):,}   Words: {len(text.split()):,}")

In [ ]:
print(stories[0])

### 🧠 Think like an LLM
*"Once upon a ____"*: you predicted **time** instantly. Why? Because you have seen that pattern many times.
The simplest possible "language model" just **counts** what usually comes next:

In [ ]:
words = text.split()
next_words = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    next_words[a][b] += 1

def show_next(word, k=6):
    counts = next_words[word]
    total = sum(counts.values())
    print(f"After '{word}':", ",  ".join(f"{w} {c / total:.0%}" for w, c in counts.most_common(k)))

show_next("upon")
show_next("little")
show_next("was")

In [ ]:
# Generate text by rolling weighted dice with these counts (a "bigram language model")
def counting_lm(start="Once", n=30):
    out = [start]
    for _ in range(n):
        options = next_words[out[-1]]
        if not options:
            break
        out.append(random.choices(list(options), weights=list(options.values()))[0])
    return " ".join(out)

print(counting_lm())

Each pair of words looks fine, but the whole thing makes no sense, because the model only ever looks at **one** previous word.
👉 GPT does the *same job* (predict the next token) but with a neural network that looks at the **whole context**.
That's what we build today.

---
# Part 2 · Tokenization: turning text into numbers

Neural networks only understand numbers. A **tokenizer** converts text ⇄ a list of integers (**token IDs**).

| Approach | Vocabulary | Problem |
|---|---|---|
| Characters | ~80 | sequences become very long; each token carries little meaning |
| Words | 100,000s | huge vocabulary; a new word ("Rahul", "ChatGPT") is unknown |
| **Subwords (BPE)** ✅ | 1,000 – 200,000 | best of both. Used by GPT, Llama, Gemini, Claude |

In [ ]:
# Option 1: character-level
chars = sorted(set(text.replace(EOT, "")))
stoi = {c: i for i, c in enumerate(chars)}
print(len(chars), "characters:", repr("".join(chars)))
print("'hello' ->", [stoi[c] for c in "hello"])

# Option 2: word-level
word_vocab = Counter(re.findall(r"\w+|[^\w\s]", text))
print("\nUnique words:", len(word_vocab))
print("Is 'Rahul' in the word vocabulary?", "Rahul" in word_vocab, " ← a word model would be stuck")

## Byte-Pair Encoding (BPE): the algorithm behind GPT's tokenizer
1. Start with single characters.
2. Count every pair of neighbouring symbols.
3. **Merge the most frequent pair** into a new symbol.
4. Repeat. Frequent words become one token; rare words are built from pieces.

Let's run it by hand on a toy corpus: `hug ×10, pug ×5, pun ×12, bun ×4, hugs ×5`

In [ ]:
def merge_pair(symbols, pair, new_symbol):
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(new_symbol)
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return out

toy_corpus = {"hug": 10, "pug": 5, "pun": 12, "bun": 4, "hugs": 5}
splits = {w: list(w) for w in toy_corpus}          # start: every word = list of characters

for step in range(1, 4):
    pair_counts = Counter()
    for word, freq in toy_corpus.items():
        s = splits[word]
        for pair in zip(s, s[1:]):
            pair_counts[pair] += freq
    best = max(pair_counts, key=lambda p: pair_counts[p])
    print(f"Step {step}: pair counts {dict(pair_counts)}")
    print(f"        → merge {best} into '{best[0] + best[1]}'")
    splits = {w: merge_pair(s, best, best[0] + best[1]) for w, s in splits.items()}
    print(f"        words are now: {splits}\n")

### Our full tokenizer
Same algorithm, plus three practical details that real tokenizers also have:
* **pre-tokenization**: split text into words first, and keep the leading space (`" cat"`) so merges never cross words
* **special tokens**: `<|endoftext|>` (end of a document) and `<unk>` (unknown character)
* a **cache**, so each unique word is encoded only once

In [ ]:
EOT = "<|endoftext|>"   # special token: "this document ends here"

UNK = "<unk>"           # special token: "character I have never seen"

SPECIAL_TOKENS = [EOT, UNK]

SPLIT_PATTERN = re.compile(
    r"'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"
)

def merge_pair(symbols, pair, new_symbol):
    """Replace every occurrence of `pair` in the list `symbols` with `new_symbol`."""
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(new_symbol)
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return out

class BPETokenizer:
    def __init__(self):
        self.merges = []      # learned merge rules, in the order they were learned
        self.vocab = []       # token id -> token string
        self.stoi = {}        # token string -> token id
        self._cache = {}      # chunk -> token ids (encoding the same word twice is wasted work)

    # ------------------------------------------------------------------ training
    def train(self, text, num_merges=500, verbose=False):
        text = text.replace(EOT, "\n")
        # Count each unique chunk once, with its frequency. This is what makes
        # training fast: TinyStories has ~1M chunks but only ~15k unique ones.
        chunk_freq = Counter(SPLIT_PATTERN.findall(text))
        splits = {chunk: list(chunk) for chunk in chunk_freq}
        alphabet = sorted({ch for chunk in chunk_freq for ch in chunk})

        self.merges = []
        for step in range(num_merges):
            # (a) count all neighbouring pairs, weighted by how often the chunk appears
            pair_counts = Counter()
            for chunk, freq in chunk_freq.items():
                symbols = splits[chunk]
                for pair in zip(symbols, symbols[1:]):
                    pair_counts[pair] += freq
            if not pair_counts:
                break
            # (b) the most frequent pair becomes a new token
            best = max(pair_counts, key=lambda pair: pair_counts[pair])
            new_symbol = best[0] + best[1]
            self.merges.append(best)
            # (c) apply the merge everywhere
            for chunk, symbols in splits.items():
                if len(symbols) > 1 and best[0] in symbols:
                    splits[chunk] = merge_pair(symbols, best, new_symbol)
            if verbose and (step < 10 or (step + 1) % 100 == 0):
                print(f"merge {step + 1:4d}: {best[0]!r:>10} + {best[1]!r:<10} -> {new_symbol!r}"
                      f"   (seen {pair_counts[best]:,} times)")

        self._build_vocab(alphabet)
        return self

    def _build_vocab(self, alphabet):
        self.alphabet = list(alphabet)
        self.vocab = SPECIAL_TOKENS + self.alphabet + [a + b for a, b in self.merges]
        self.stoi = {tok: i for i, tok in enumerate(self.vocab)}
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self._cache = {}

    @property
    def vocab_size(self):
        return len(self.vocab)

    @property
    def eot_id(self):
        return self.stoi[EOT]

    # ------------------------------------------------------------------ encoding
    def _encode_chunk(self, chunk):
        if chunk in self._cache:
            return self._cache[chunk]
        symbols = [ch if ch in self.stoi else UNK for ch in chunk]
        # Apply learned merges in the SAME order they were learned (lowest rank first)
        while len(symbols) > 1:
            pairs = list(zip(symbols, symbols[1:]))
            best = min(pairs, key=lambda p: self.ranks.get(p, float("inf")))
            if best not in self.ranks:
                break
            symbols = merge_pair(symbols, best, best[0] + best[1])
        ids = [self.stoi[s] for s in symbols]
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        parts = text.split(EOT)
        for i, part in enumerate(parts):
            for chunk in SPLIT_PATTERN.findall(part):
                ids.extend(self._encode_chunk(chunk))
            if i < len(parts) - 1:
                ids.append(self.eot_id)
        return ids

    def decode(self, ids):
        return "".join(self.vocab[i] for i in ids)

    def tokens(self, text):
        """Human-readable view: the list of token strings for `text`."""
        return [self.vocab[i] for i in self.encode(text)]

    # ------------------------------------------------------------------ save / load
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"alphabet": self.alphabet, "merges": self.merges}, f)

    @classmethod
    def load(cls, path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.merges = [tuple(pair) for pair in data["merges"]]
        tok._build_vocab(data["alphabet"])
        return tok

In [ ]:
t0 = time.time()
tokenizer = BPETokenizer().train(text, num_merges=1000, verbose=True)
print(f"\nVocabulary size: {tokenizer.vocab_size}   (trained in {time.time() - t0:.0f} s)")
print("Some learned tokens:", tokenizer.vocab[-15:])

In [ ]:
def show_tokens(pieces):
    colours = ["#bfdbfe", "#fde68a", "#bbf7d0", "#fecaca", "#ddd6fe"]
    spans = "".join(
        f'<span style="background:{colours[i % 5]};color:#111;padding:2px 1px;margin:1px;'
        f'border-radius:3px;font-family:monospace;white-space:pre">{html.escape(p).replace(chr(10), "↵")}</span>'
        for i, p in enumerate(pieces))
    display(HTML(f'<div style="line-height:2.2">{spans}</div><small>{len(pieces)} tokens</small>'))

sentence = "Once upon a time, there was a little girl named Lily. She was unbelievably happy!"
show_tokens(tokenizer.tokens(sentence))
ids = tokenizer.encode(sentence)
print("Token IDs:", ids)
print("Decoded  :", tokenizer.decode(ids))

In [ ]:
# Encode the whole dataset: from now on the model only sees these numbers
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
assert tokenizer.decode(data[:2000].tolist()) == text[:len(tokenizer.decode(data[:2000].tolist()))]
print(f"{len(text):,} characters → {len(data):,} tokens  ({len(text) / len(data):.2f} characters per token)")

### Compare with the real GPT-4o tokenizer
`tiktoken` is OpenAI's open-source tokenizer library. It runs **on our machine**; no API or key involved.

In [ ]:
import tiktoken
gpt4o = tiktoken.get_encoding("o200k_base")    # GPT-4o's tokenizer: 200k vocabulary
gpt4 = tiktoken.get_encoding("cl100k_base")    # GPT-4's tokenizer:  100k vocabulary

samples = {
    "English":  "I love learning about artificial intelligence.",
    "Hindi":    "मुझे आर्टिफिशियल इंटेलिजेंस के बारे में सीखना पसंद है।",
    "Hinglish": "Mujhe artificial intelligence ke baare mein seekhna pasand hai.",
    "Code":     "for i in range(10): print(i)",
}
print(f"{'':9s} {'chars':>6s} {'GPT-4':>6s} {'GPT-4o':>7s} {'ours':>5s}")
for name, s in samples.items():
    print(f"{name:9s} {len(s):6d} {len(gpt4.encode(s)):6d} {len(gpt4o.encode(s)):7d} {len(tokenizer.encode(s)):5d}")

print("\nGPT-4o splits the Hindi sentence like this:")
show_tokens([gpt4o.decode([t]) for t in gpt4o.encode(samples["Hindi"])])

In [ ]:
# Why do LLMs struggle to count the r's in "strawberry"? They never see letters, only tokens!
for w in ["strawberry", " strawberry", "Strawberry"]:
    print(f"{w!r:15} → {[gpt4o.decode([t]) for t in gpt4o.encode(w)]}")

**Why tokens matter in industry**
* 💰 **Cost**: APIs charge per million tokens, so Hindi text costs more than English for the same meaning.
* 📏 **Context window**: a model can only "see" a fixed number of tokens (ours: 128; modern LLMs: 128k–1M+).
* ⚡ **Speed**: the model generates **one token at a time**.

✍️ **Your turn:** tokenize your own name and a Hinglish sentence with both tokenizers. Which one uses fewer tokens?

In [ ]:
my_text = "Mera naam Rahul hai aur mujhe AI pasand hai"   # ← change me
show_tokens(tokenizer.tokens(my_text))
show_tokens([gpt4o.decode([t]) for t in gpt4o.encode(my_text)])

---
# Part 3 · Embeddings: giving tokens a meaning

A token ID (e.g. `417`) is just a label, and `417` is not "closer" to `418` in meaning.
So every token gets a **vector** (a list of numbers) called its **embedding**. Similar meanings → nearby vectors.

Let's build embeddings **by hand** first, with features we choose ourselves:

In [ ]:
features = ["royal", "male", "female", "animal", "fruit"]
toy = {
    "king":  [1.0, 0.5, 0.0, 0.0, 0.0],
    "queen": [1.0, 0.0, 0.5, 0.0, 0.0],
    "man":   [0.1, 0.5, 0.0, 0.0, 0.0],
    "woman": [0.1, 0.0, 0.5, 0.0, 0.0],
    "lion":  [0.6, 0.4, 0.0, 1.0, 0.0],   # "king of the jungle"
    "mango": [0.5, 0.0, 0.0, 0.0, 1.0],   # "king of fruits"
}
toy = {w: np.array(v) for w, v in toy.items()}

def cosine(a, b):
    # 1 = same direction (similar meaning), 0 = unrelated
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

for w in ["queen", "man", "lion", "woman", "mango"]:
    print(f"similarity(king, {w:5s}) = {cosine(toy['king'], toy[w]):.2f}")

In [ ]:
# Vector arithmetic: king - man + woman = ?
result = toy["king"] - toy["man"] + toy["woman"]
ranking = sorted(((cosine(result, v), w) for w, v in toy.items() if w not in ("king", "man", "woman")), reverse=True)
print("king - man + woman ≈", ranking[0][1], f"(similarity {ranking[0][0]:.2f})")

In a real model nobody hand-writes these features. The embedding table starts **random** and the model
**learns** the numbers during training. GPT-3 uses 12,288 numbers per token; we will use 128.

In PyTorch the embedding layer is simply a **lookup table**: row *i* is the vector of token *i*.

In [ ]:
emb = nn.Embedding(num_embeddings=tokenizer.vocab_size, embedding_dim=8)
print("Embedding table shape:", tuple(emb.weight.shape), "= (vocab size, vector size)")

dog, cat, happy = (tokenizer.encode(w)[0] for w in [" dog", " cat", " happy"])
print("token id of ' dog':", dog)
print("vector of ' dog'  :", emb(torch.tensor(dog)).detach().numpy().round(2))
print("row", dog, "of table :", emb.weight[dog].detach().numpy().round(2), "← identical: it IS a lookup")

sim = lambda a, b: F.cosine_similarity(emb.weight[a], emb.weight[b], dim=0).item()
print(f"\nUNTRAINED: sim(dog, cat) = {sim(dog, cat):+.2f}   sim(dog, happy) = {sim(dog, happy):+.2f}   ← random!")

**Position matters too.** *"dog bites man"* and *"man bites dog"* contain the same tokens.
So GPT adds a second, learned **position embedding** (one vector per position 0, 1, 2, …) to every token embedding:

`input to the transformer = token_embedding[token] + position_embedding[position]`

We'll come back after training to see what our model learned (Part 6). 🔮

---
# Part 4 · The Transformer: building GPT piece by piece

### 4.1 How the training examples look
One chunk of text gives us many examples: every prefix predicts its next token.

In [ ]:
x, y = data[:8], data[1:9]
for t in range(8):
    context = tokenizer.decode(x[:t + 1].tolist())
    print(f"{context!r:48} → predict {tokenizer.decode([y[t].item()])!r}")

In [ ]:
BLOCK_SIZE = 128 if device == "cuda" else 64     # context window of our model (tokens)
BATCH_SIZE = 32

n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]        # 90% to learn from, 10% to check on unseen text

def get_batch(split, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    xb = torch.stack([src[i:i + block_size] for i in ix])
    yb = torch.stack([src[i + 1:i + block_size + 1] for i in ix])     # shifted by one token
    return xb.to(device), yb.to(device)

xb, yb = get_batch("train")
print("inputs :", tuple(xb.shape), "= (batch, time)")
print("targets:", tuple(yb.shape))

### 4.2 Baseline: a model with NO context (bigram)
It predicts the next token from the **current token only**. Let's see how far that gets us.

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)   # row i = scores for the token after token i

    def forward(self, idx, targets=None):
        logits = self.table(idx)                            # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

@torch.no_grad()
def quick_generate(model, prompt, max_new_tokens=40, block_size=BLOCK_SIZE):
    idx = torch.tensor([tokenizer.encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return tokenizer.decode(idx[0].tolist())

bigram = BigramLM(tokenizer.vocab_size).to(device)
optimizer = torch.optim.AdamW(bigram.parameters(), lr=1e-2)
for step in range(501):
    xb, yb = get_batch("train")
    _, loss = bigram(xb, yb)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.3f}")
print("\n" + quick_generate(bigram, "Once upon a time"))

Better than random, but it has **no memory of what came before**. Tokens need a way to *talk to each other*.
That mechanism is **self-attention**, the heart of the Transformer ("Attention Is All You Need", Google, 2017).

### 4.3 Self-attention from scratch
> *"The animal didn't cross the street because **it** was too tired."*
> To understand **it**, the model must look back at **animal**. Attention lets every token decide
> **which earlier tokens to look at, and how much**.

**Step 1: the simplest version: each token takes the average of itself and all previous tokens.**

In [ ]:
T = 4
x = torch.tensor([[1., 0.], [0., 1.], [1., 1.], [3., 3.]])   # 4 tokens, each a 2-number vector

weights = torch.tril(torch.ones(T, T))              # lower triangle: only look at the past
weights = weights / weights.sum(dim=1, keepdim=True)  # each row sums to 1
print("attention weights:\n", weights)
print("\noutput = weights @ x:\n", weights @ x, "\n← row 4 is the average of all 4 tokens")

In [ ]:
# Step 2: the same result written with softmax + a mask (-inf → probability 0).
# This is exactly how real attention is written.
scores = torch.zeros(T, T)
scores = scores.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))
print(F.softmax(scores, dim=-1))

**Step 3: real self-attention: the scores are *learned*, not all equal.**
Every token produces three vectors (think of a library 📚):
* **Query (Q)**: "what am I looking for?" (your search question)
* **Key (K)**: "what do I contain?" (the label on each book)
* **Value (V)**: "what I give you if you pick me" (the content of the book)

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$$

In [ ]:
torch.manual_seed(42)
B, T, C, head_size = 1, 6, 16, 8
x = torch.randn(B, T, C)                     # 6 tokens, 16 numbers each (pretend embeddings)

query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
q, k, v = query(x), key(x), value(x)

scores = q @ k.transpose(-2, -1) / head_size ** 0.5                       # how well each query matches each key
scores = scores.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))   # no peeking at the future
weights = F.softmax(scores, dim=-1)                                     # turn scores into percentages
out = weights @ v                                                       # mix the values

print("attention weights (row = token, columns = which past tokens it looks at):")
print(weights[0].detach().numpy().round(2))
print("\noutput shape:", tuple(out.shape))

### 4.4 The full GPT
Now we stack the pieces exactly like GPT-2:
* **Multi-head attention**: several attention heads in parallel, each learns a different relationship
* **Feed-forward (MLP)**: after "talking", each token "thinks" on its own
* **Residual connections + LayerNorm**: keep training stable in deep networks
* **Block × N**: repeat. GPT-2 small has 12 blocks, GPT-3 has 96. We use 4.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 1079
    block_size: int = 128     # context window: how many tokens the model can "see"
    n_embd: int = 128         # size of each token's embedding vector
    n_head: int = 4           # attention heads per layer
    n_layer: int = 4          # transformer blocks
    dropout: float = 0.1

class Head(nn.Module):
    """One head of masked self-attention.

    Every token produces three vectors:
      query (Q): "what am I looking for?"
      key   (K): "what do I contain?"
      value (V): "what will I share if you pick me?"
    Attention(Q, K, V) = softmax(Q·Kᵀ / √d) · V
    """

    def __init__(self, cfg, head_size):
        super().__init__()
        self.key = nn.Linear(cfg.n_embd, head_size, bias=False)
        self.query = nn.Linear(cfg.n_embd, head_size, bias=False)
        self.value = nn.Linear(cfg.n_embd, head_size, bias=False)
        # lower-triangular mask: a token may look at itself and the past, never the future
        self.register_buffer("tril", torch.tril(torch.ones(cfg.block_size, cfg.block_size)),
                             persistent=False)
        self.dropout = nn.Dropout(cfg.dropout)
        self.last_attention = None   # saved so the app can visualise it

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)                                            # (B, T, hs)
        k = self.key(x)                                              # (B, T, hs)
        scores = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5       # (B, T, T) similarity
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # hide the future
        weights = F.softmax(scores, dim=-1)                          # each row sums to 1
        self.last_attention = weights.detach()
        weights = self.dropout(weights)
        v = self.value(x)                                            # (B, T, hs)
        return weights @ v                                           # weighted mix of values

class MultiHeadAttention(nn.Module):
    """Several heads in parallel: each head can learn a different kind of relationship."""

    def __init__(self, cfg):
        super().__init__()
        head_size = cfg.n_embd // cfg.n_head
        self.heads = nn.ModuleList([Head(cfg, head_size) for _ in range(cfg.n_head)])
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """A small MLP applied to every token on its own: 'think about what I just gathered'."""

    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Transformer block: communication (attention) followed by computation (MLP)."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.ffwd = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual connection ("skip connection")
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.n_embd)     # WHAT the token is
        self.position_embedding = nn.Embedding(cfg.block_size, cfg.n_embd)  # WHERE the token is
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size)                # scores for next token
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding(idx)                                      # (B, T, C)
        pos = self.position_embedding(torch.arange(T, device=idx.device))    # (T, C)
        x = self.blocks(tok + pos)
        logits = self.lm_head(self.ln_f(x))                                  # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def attention_maps(self):
        """[layer][head] -> (T, T) attention matrix from the most recent forward pass."""
        return [[h.last_attention[0] for h in block.attn.heads] for block in self.blocks]

def save_checkpoint(model, path, extra=None):
    torch.save({"config": asdict(model.cfg), "model": model.state_dict(), "extra": extra or {}}, path)

def load_checkpoint(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=True)  # tensors + plain data only
    model = MiniGPT(GPTConfig(**ckpt["config"])).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, ckpt.get("extra", {})

In [ ]:
if device == "cuda":
    cfg = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=128, n_embd=128, n_head=4, n_layer=4)
    MAX_ITERS, EVAL_EVERY = 3000, 300
else:   # smaller and faster so that CPU-only laptops can finish in a few minutes
    cfg = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=64, n_embd=96, n_head=4, n_layer=3)
    MAX_ITERS, EVAL_EVERY = 1000, 200

model = MiniGPT(cfg).to(device)
print(f"Our GPT has {model.num_params():,} parameters")
print(cfg)

xb, yb = get_batch("train", cfg.block_size)
logits, loss = model(xb, yb)
print(f"\nlogits shape: {tuple(logits.shape)} = (batch, time, vocab): a score for EVERY possible next token")
print(f"loss before training: {loss.item():.2f}  (pure guessing = ln({cfg.vocab_size}) = {math.log(cfg.vocab_size):.2f})")

---
# Part 5 · Training (pre-training)

Every step:
1. **Forward**: the model predicts the next token at every position
2. **Loss**: cross-entropy = how *surprised* the model was by the real next token
3. **Backward**: backpropagation computes how each of the ~1M weights should change
4. **Step**: the AdamW optimizer nudges the weights

Watch the sample text go from gibberish → words → sentences. ⏱️ ~2 min on GPU, ~5 min on CPU.

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            _, loss = model(*get_batch(split, cfg.block_size))
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = []
t0 = time.time()
for step in range(MAX_ITERS + 1):
    if step % EVAL_EVERY == 0 or step == MAX_ITERS:
        losses = estimate_loss(model)
        history.append({"step": step, **losses})
        sample = quick_generate(model, "Once upon a time", 25, cfg.block_size).replace("\n", " ")
        print(f"step {step:5d} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f} "
              f"| {time.time() - t0:4.0f}s | {sample}")
    if step == MAX_ITERS:
        break
    xb, yb = get_batch("train", cfg.block_size)
    _, loss = model(xb, yb)                  # 1. forward + 2. loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()                          # 3. backpropagation
    optimizer.step()                         # 4. update weights
model.eval()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot([h["step"] for h in history], [h["train"] for h in history], "o-", label="train loss")
plt.plot([h["step"] for h in history], [h["val"] for h in history], "o-", label="validation loss")
plt.xlabel("training step"); plt.ylabel("cross-entropy loss"); plt.title("Our GPT is learning"); plt.legend()
plt.show()

In [ ]:
# Save our model, like every AI team saves "checkpoints"
save_checkpoint(model, "storygpt.pt", {"history": history, "iters": MAX_ITERS, "train_tokens": len(train_data),
                                        "train_seconds": round(time.time() - t0), "device": device})
tokenizer.save("tokenizer.json")
print("Saved storygpt.pt and tokenizer.json (Colab: see the 📁 Files panel on the left)")

<details><summary>🆘 <b>Backup plan</b>: training too slow or crashed? Click here.</summary>

Upload `storygpt.pt` and `tokenizer.json` (shared by your instructor) into the 📁 Files panel, then run:
```python
model, _ = load_checkpoint("storygpt.pt", device)
tokenizer = BPETokenizer.load("tokenizer.json")
cfg = model.cfg
```
</details>

---
# Part 6 · Look inside the trained model 🔬

### Embeddings: what did it learn about meaning?
Nobody told the model that *dog* and *cat* are similar. Let's check what it learned by itself.

In [ ]:
E = model.token_embedding.weight.detach().cpu()
E = E / E.norm(dim=1, keepdim=True)                  # normalise → dot product = cosine similarity

def word_vector(word):
    ids = tokenizer.encode(" " + word)
    return E[ids].mean(0)

def nearest(word, k=8):
    v = word_vector(word)
    sims = E @ (v / v.norm())
    out = []
    for i in torch.argsort(sims, descending=True).tolist():
        tok = tokenizer.vocab[i].strip()
        if tok.isalpha() and tok != word:
            out.append(f"{tok} ({sims[i]:.2f})")
        if len(out) == k:
            break
    print(f"{word:>6} → " + ", ".join(out))

for w in ["dog", "happy", "red", "mom", "ball", "ran"]:
    nearest(w)

In [ ]:
words_to_plot = ["dog", "cat", "bird", "fish", "happy", "sad", "angry", "red", "blue", "green",
                 "mom", "dad", "sister", "ball", "toy", "car", "ran", "jumped", "walked"]
V = torch.stack([word_vector(w) for w in words_to_plot]).numpy()
V = V - V.mean(0)
_, _, vt = np.linalg.svd(V, full_matrices=False)     # PCA: squash 128 numbers down to 2 for plotting
xy = V @ vt[:2].T
plt.figure(figsize=(8, 6))
plt.scatter(xy[:, 0], xy[:, 1])
for (px, py), w in zip(xy, words_to_plot):
    plt.annotate(w, (px, py), xytext=(4, 4), textcoords="offset points", fontsize=12)
plt.title("Embedding map learned by OUR model"); plt.xticks([]); plt.yticks([])
plt.show()

### Attention: which words does each word look at?

In [ ]:
def plot_attention(sentence, layer, head):
    ids = tokenizer.encode(sentence)[-cfg.block_size:]
    with torch.no_grad():
        model(torch.tensor([ids], device=device))
    att = model.attention_maps()[layer][head].cpu().numpy()
    labels = [tokenizer.vocab[i] for i in ids]
    plt.figure(figsize=(9, 8))
    plt.imshow(att, cmap="Blues")
    plt.xticks(range(len(ids)), labels, rotation=90); plt.yticks(range(len(ids)), labels)
    plt.xlabel("... looks at this earlier token"); plt.ylabel("This token ...")
    plt.title(f"Layer {layer + 1}, head {head + 1}")
    plt.show()

plot_attention("Lily had a red ball. She gave the ball to her friend Tom.", layer=cfg.n_layer - 1, head=0)

✍️ **Your turn:** try other layers/heads. Some heads look at the previous word, some look back at names, some at punctuation.

### What is the model "thinking"? Next-token probabilities

In [ ]:
@torch.no_grad()
def plot_next_token(prompt, k=10):
    ids = tokenizer.encode(prompt)[-cfg.block_size:]
    logits, _ = model(torch.tensor([ids], device=device))
    probs = F.softmax(logits[0, -1], dim=-1).cpu()
    top = torch.topk(probs, k)
    labels = [repr(tokenizer.vocab[i]) for i in top.indices.tolist()]
    plt.figure(figsize=(8, 3.5))
    plt.barh(labels[::-1], top.values.numpy()[::-1])
    plt.title(f"Next token after: {prompt!r}"); plt.xlabel("probability")
    plt.show()

plot_next_token("Once upon a time, there was a little")
plot_next_token("Lily was very sad because she lost her")

---
# Part 7 · Generation & prompting

The model gives **probabilities**. *How we choose* the next token is a design decision: the same
`temperature`, `top_k`, `top_p` settings you see in every LLM product.

| Setting | What it does | Effect |
|---|---|---|
| **Greedy** (temperature 0) | always pick the most likely token | safe but boring, often repeats itself |
| **Temperature** | divide scores by T before softmax | T < 1 focused · T > 1 creative/chaotic |
| **Top-k** | only the k most likely tokens may be picked | cuts off the crazy tail |
| **Top-p** (nucleus) | smallest set of tokens whose probabilities add up to p | adapts to how sure the model is |

In [ ]:
def apply_temperature(logits, temperature):
    return logits / max(temperature, 1e-6)

def top_k_filter(logits, k):
    """Everything outside the k highest logits gets -inf (probability 0)."""
    if k is None or k <= 0 or k >= logits.size(-1):
        return logits
    kth_best = torch.topk(logits, k).values[..., -1, None]
    return logits.masked_fill(logits < kth_best, float("-inf"))

def top_p_filter(logits, p):
    """Nucleus sampling: keep the smallest set of tokens whose total probability >= p."""
    if p is None or p >= 1.0:
        return logits
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative - F.softmax(sorted_logits, dim=-1) >= p   # always keeps the top token
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    return torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=150, temperature=0.8,
             top_k=None, top_p=None, stop_at_eot=True):
    """Generate text and return a list of steps: (token_text, probability_of_chosen_token).

    The probability is how confident the model was about the token it produced.
    We use it later as a simple 'hallucination radar'.
    """
    model.eval()
    device = next(model.parameters()).device
    ids = tokenizer.encode(prompt) or [tokenizer.eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    block_size = model.cfg.block_size
    steps = []
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]                  # the model can only see block_size tokens
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]                        # we only need the LAST position
        raw_probs = F.softmax(logits, dim=-1)            # the model's honest opinion
        if temperature == 0:                             # greedy decoding
            next_id = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            logits = apply_temperature(logits, temperature)
            logits = top_k_filter(logits, top_k)
            logits = top_p_filter(logits, top_p)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)   # roll the weighted dice
        confidence = raw_probs[0, next_id.item()].item()
        if stop_at_eot and next_id.item() == tokenizer.eot_id:
            break
        steps.append((tokenizer.decode([next_id.item()]), confidence))
        idx = torch.cat([idx, next_id], dim=1)
    return steps

def generate_text(model, tokenizer, prompt, **kwargs):
    return prompt + "".join(tok for tok, _ in generate(model, tokenizer, prompt, **kwargs))

@torch.no_grad()
def next_token_table(model, tokenizer, prompt, k=10):
    """Top-k candidates for the very next token: what the model is 'thinking'."""
    model.eval()
    device = next(model.parameters()).device
    ids = tokenizer.encode(prompt)[-model.cfg.block_size:]
    logits, _ = model(torch.tensor([ids], dtype=torch.long, device=device))
    probs = F.softmax(logits[0, -1], dim=-1)
    top = torch.topk(probs, k)
    return [(tokenizer.vocab[i], p) for i, p in zip(top.indices.tolist(), top.values.tolist())]

In [ ]:
# See what temperature does to the probability distribution
ids = tokenizer.encode("Once upon a time, there was a little")
with torch.no_grad():
    logits = model(torch.tensor([ids], device=device))[0][0, -1].cpu()
top = torch.topk(logits, 12).indices
labels = [tokenizer.vocab[i].strip() for i in top.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 2.0]):
    probs = F.softmax(apply_temperature(logits, T), dim=-1)[top]
    ax.bar(labels, probs.numpy()); ax.set_title(f"temperature = {T}"); ax.tick_params(axis="x", rotation=70)
plt.tight_layout(); plt.show()

In [ ]:
prompt = "Once upon a time, there was a little dog named"
print("🥶 GREEDY (T=0), run twice:")
for _ in range(2):
    print("  ", generate_text(model, tokenizer, prompt, max_new_tokens=40, temperature=0).replace("\n", " "))
print("\n🙂 T=0.8, top-k=40:")
for _ in range(2):
    print("  ", generate_text(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8, top_k=40).replace("\n", " "))
print("\n🤪 T=2.0 (too hot):")
print("  ", generate_text(model, tokenizer, prompt, max_new_tokens=40, temperature=2.0).replace("\n", " "))

### Prompting a *base* model
Our model is a **base model**: it only continues text. The prompt is simply *the beginning of the document*.
So good prompting = writing the start of the text you want.

In [ ]:
for p in ["Once upon a time, there was a brave little girl named Asha. She",
          "It was a rainy day. Tom was sad because",
          "The cat and the dog were best friends. One day, they"]:
    print("▶", generate_text(model, tokenizer, p, max_new_tokens=50, temperature=0.8, top_k=40).replace("\n", " "), "\n")

In [ ]:
# But try giving it an INSTRUCTION, like you would give ChatGPT:
print(generate_text(model, tokenizer, "Write a story about a cat named Rahul.\n",
                    max_new_tokens=50, temperature=0.8, top_k=40))

It ignores the instruction because it was never taught what an instruction is.
This is exactly the difference between **GPT-3 (base, 2020)** and **ChatGPT (2022)**:

```
① Pre-training       predict next token on trillions of tokens       → knows language   (we did this: Part 5)
② Instruction tuning train on (instruction → good answer) examples    → follows requests (next: Part 8)
③ RLHF / preference  humans rank answers, model learns to prefer good  → helpful & safe
```

---
# Part 8 · Instruction tuning (SFT): turning our base model into an assistant

We create an **instruction dataset** automatically: for every story that contains *"a ___ named ___"*,
we write the instruction that could have produced it. Then we **fine-tune** our model on this chat format:

```
User: Write a story about a dog named Max.
Assistant: <the story>
```

This `User: … Assistant:` layout is a **chat template**. ChatGPT, Llama and Gemini all use one
(with special tokens like `<|im_start|>`). When you "chat", the app is really just building this text.

**Data design trick.** If we use the stories as they are, the model mostly ignores the name in the request
(we measured: ~25% success). So we **swap the name** in every story for a random one, half real names and half
made-up names like *"Kovira"*. A made-up name can't be memorised, so the only way to get the story right is
to **copy the name from the request**. Real AI teams spend most of their fine-tuning effort on data like this.

In [ ]:
PATTERN = re.compile(r"\b[Aa]n? (?:(?:little|big|small|young|old|happy|cute|kind|friendly|brave|curious|tiny|"
                     r"fluffy|smart|silly|good|nice|shy|lazy|pretty|wise|grumpy|playful)\s)*([a-z]+) named ([A-Z][a-z]+)")

NAMES = """Aarav Aditi Aditya Akash Akshay Amit Ananya Anika Anjali Ankit Anushka Arjun Arnav Aryan Asha Ayaan
Bhavya Chetan Chirag Deepak Dev Dhruv Diya Divya Gaurav Gauri Harsh Isha Ishaan Jatin Jiya Kabir Karan Kavya Khushi
Krish Kriti Lakshya Manav Manish Meera Mehul Mohit Muskan Naina Nandini Naveen Neel Nikhil Nisha Om Pallavi Parth
Pooja Pranav Prateek Priya Radha Rahul Raj Rajat Riya Rohan Rohit Ruchi Sahil Saloni Sameer Sanjay Sanya Shreya
Shruti Simran Sneha Sonia Suresh Tanvi Tanya Tushar Uday Vanya Varun Vedant Vikram Vivek Yash Zara
Max Lily Tom Mia Leo Zoe Sam Nina Ben Emma Jack Lucy Noah Ella Finn Ruby Oscar Grace Henry Alice""".split()

ANIMALS = ["dog", "cat", "bird", "fish", "rabbit", "bunny", "bear", "mouse", "duck", "frog", "lion", "monkey",
           "elephant", "horse", "cow", "owl", "puppy", "kitten", "turtle", "squirrel"]

def chat_format(instruction, answer=""):
    return f"User: {instruction}\nAssistant: {answer}"

def a_or_an(noun):
    return "an" if noun[0] in "aeiou" else "a"

def random_name():
    """A made-up name like 'Kovira'. The model can't have memorised it, so it must learn to copy."""
    syllables = [c + v for c in "bdghjklmnprstvyz" for v in ["a", "e", "i", "o", "u", "aa", "ee"]]
    return "".join(random.choice(syllables) for _ in range(random.randint(2, 3))).capitalize()

def build_sft_examples(stories):
    """Turn raw stories into (instruction -> story) chat examples, with names/animals swapped at random."""
    examples = []
    for story in stories:
        m = PATTERN.search(story)
        if not m:
            continue
        noun, name = m.group(1), m.group(2)
        new_name = random.choice(NAMES) if random.random() < 0.5 else random_name()
        story = re.sub(rf"\b{name}\b", new_name, story)
        if noun in ANIMALS:
            new_noun = random.choice(ANIMALS)
            story = re.sub(rf"\b{noun}\b", new_noun, story)
            noun = new_noun
        examples.append(chat_format(f"Write a story about {a_or_an(noun)} {noun} named {new_name}.", story))
    return examples

In [ ]:
random.seed(0)
sft_examples = build_sft_examples(stories)
print(len(sft_examples), "instruction examples. Two of them:\n")
for e in sft_examples[:2]:
    print(e[:220], "...\n")

In [ ]:
# Fine-tuning = continue training the SAME model on the new examples (smaller learning rate)
sft_ids = [tokenizer.encode(e) + [tokenizer.eot_id] for e in sft_examples]
sft_ids = [e for e in sft_ids if len(e) > cfg.block_size + 1]

def get_sft_batch(batch_size=BATCH_SIZE):
    rows = random.sample(sft_ids, batch_size)          # every row starts at "User: ..."
    xb = torch.tensor([r[:cfg.block_size] for r in rows])
    yb = torch.tensor([r[1:cfg.block_size + 1] for r in rows])
    return xb.to(device), yb.to(device)

sft_model = copy.deepcopy(model).train()
optimizer = torch.optim.AdamW(sft_model.parameters(), lr=5e-4)
for step in range(801):
    _, loss = sft_model(*get_sft_batch())
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 200 == 0:
        print(f"fine-tune step {step:3d}  loss {loss.item():.3f}")
sft_model.eval()
save_checkpoint(sft_model, "storygpt_sft.pt", {"history": history, "iters": MAX_ITERS, "sft_steps": 800})

In [ ]:
request = "Write a story about a cat named Rahul."        # ← try your own name and an animal!
print("🟥 BASE model:\n", generate_text(model, tokenizer, chat_format(request), max_new_tokens=60,
                                        temperature=0.8, top_k=40), "\n")
print("🟩 INSTRUCTION-TUNED model:\n", generate_text(sft_model, tokenizer, chat_format(request),
                                                    max_new_tokens=60, temperature=0.8, top_k=40))

In [ ]:
# How often does it follow the request? Names it saw in fine-tuning vs names it NEVER saw
def follows(request_noun, request_name, tries=4):
    ok = 0
    for _ in range(tries):
        out = generate_text(sft_model, tokenizer, chat_format(f"Write a story about {a_or_an(request_noun)} "
                            f"{request_noun} named {request_name}."), max_new_tokens=40, temperature=0.8, top_k=40)
        ok += request_name in out.split("Assistant:", 1)[1]
    return ok / tries

for noun, name in [("cat", "Rahul"), ("bird", "Priya"), ("dog", "Vishal"), ("girl", "Ishita")]:
    seen = "seen in fine-tuning" if name in NAMES else "NEVER seen"
    print(f"{name:7s} ({seen:19s}) → followed {follows(noun, name):.0%} of the time")

When a name was never seen, the model often "snaps" to the closest name it knows (*Vishal → Vivek*).
That's a small **hallucination**: it produces something familiar and plausible instead of what you asked for.

---
# Part 9 · Hallucination: fluent ≠ true

A **hallucination** is fluent, confident-sounding text that is **false or made up**.
Our model has never read a single fact about the world. Watch what it does anyway:

In [ ]:
def show_confidence(steps, prompt=""):
    # Colour every generated token by how sure the model was: green = sure, yellow = unsure, red = guessing
    def colour(p):
        return "#86efac" if p >= 0.5 else "#fde68a" if p >= 0.15 else "#fca5a5"
    spans = "".join(f'<span title="{p:.0%}" style="background:{colour(p)};color:#111;padding:1px;border-radius:3px;'
                    f'white-space:pre-wrap">{html.escape(t)}</span>' for t, p in steps)
    display(HTML(f'<div style="line-height:1.9;font-family:monospace"><b>{html.escape(prompt)}</b>{spans}</div>'))

for prompt in ["The capital of India is", "The Prime Minister of India said that", "Albert Einstein was a"]:
    show_confidence(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8, top_k=40), prompt)

It never says *"I don't know"*. It produces the **most story-like continuation**, because that is all it was
trained to do. Big LLMs hallucinate for the same reason: they generate *plausible* text, not *verified* text.

**Confidence ≠ correctness**: green tokens can still be false. Colouring by probability is a useful warning light, not a fact-checker.

| Why LLMs hallucinate | How industry reduces it |
|---|---|
| trained to be fluent, not truthful | **RAG**: give the model real documents to read (**Day 8!**) |
| no fact database, only patterns | lower temperature for factual tasks |
| knowledge cutoff (training date) | ask for sources / citations, then verify them |
| sampling randomness | fine-tune to say "I don't know"; human review for critical use |

---
# Part 10 · AI Safety: guardrails, privacy, bias

A model is never shipped "naked":

```
user input ──► INPUT GUARDRAILS ──► model ──► OUTPUT GUARDRAILS ──► user
```

In [ ]:
PII_PATTERNS = {
    "EMAIL":   re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "AADHAAR": re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"),
    "PHONE":   re.compile(r"(?<!\d)(?:\+91[\s-]?|0)?[6-9]\d{4}\s?\d{5}(?!\d)"),
    "PAN":     re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
}

def mask_pii(text):
    """Replace personal data with placeholders. Returns (masked_text, list_of_findings)."""
    findings = []
    for label, pattern in PII_PATTERNS.items():
        for match in pattern.findall(text):
            findings.append(f"{label}: {match}")
        text = pattern.sub(f"[{label}]", text)
    return text, findings

BLOCKED_WORDS = [
    "kill", "killed", "murder", "gun", "guns", "knife", "bomb", "blood", "weapon",
    "drugs", "alcohol", "suicide", "hate", "stupid", "idiot",
]

_BLOCKED_RE = re.compile(r"\b(" + "|".join(BLOCKED_WORDS) + r")\b", re.IGNORECASE)

def find_blocked(text):
    return sorted({m.lower() for m in _BLOCKED_RE.findall(text)})

def check_context(tokenizer, prompt, block_size):
    n = len(tokenizer.encode(prompt))
    if n > block_size:
        return n, f"Prompt is {n} tokens but the model only sees the last {block_size}. The start will be forgotten."
    return n, None

def _normalise(text):
    return " ".join(re.sub(r"<\|endoftext\|>", " ", text).split())

class MemorizationChecker:
    """Finds word-for-word overlap between generated text and the training data."""

    def __init__(self, training_text, n=8):
        self.corpus = _normalise(training_text)
        self.n = n

    def check(self, generated):
        words = _normalise(generated).split()
        if len(words) < self.n:
            return {"copied_fraction": 0.0, "longest_copy": ""}
        copied = [False] * len(words)
        for i in range(len(words) - self.n + 1):
            window = " ".join(words[i:i + self.n])
            if window in self.corpus:
                for j in range(i, i + self.n):
                    copied[j] = True
        # longest run of copied words
        best, start = (0, 0), None
        for i, flag in enumerate(copied + [False]):
            if flag and start is None:
                start = i
            if not flag and start is not None:
                best = max(best, (i - start, start))
                start = None
        length, begin = best
        return {"copied_fraction": sum(copied) / len(words),
                "longest_copy": " ".join(words[begin:begin + length])}

def low_confidence_tokens(steps, threshold=0.15):
    """steps = [(token_text, probability), ...] from generate(). Returns the shaky ones."""
    return [(tok, p) for tok, p in steps if p < threshold and tok.strip()]

def check_input(prompt, tokenizer=None, block_size=None):
    """Run every input guardrail. Returns (clean_prompt, blocked: bool, messages)."""
    messages = []
    clean, pii = mask_pii(prompt)
    if pii:
        messages.append("🔒 Personal data masked before it reached the model: " + ", ".join(pii))
    blocked = find_blocked(clean)
    if blocked:
        messages.append("⛔ Blocked: unsafe topic for a kids' story app (" + ", ".join(blocked) + ")")
    if tokenizer is not None and block_size is not None:
        _, warning = check_context(tokenizer, clean, block_size)
        if warning:
            messages.append("📏 " + warning)
    return clean, bool(blocked), messages

def check_output(text):
    """Run output guardrails. Returns (safe_text, messages)."""
    messages = []
    blocked = find_blocked(text)
    if blocked:
        text = _BLOCKED_RE.sub("****", text)
        messages.append("🧹 Output filter removed: " + ", ".join(blocked))
    return text, messages

In [ ]:
print(check_input("My number is 9876543210 and email is riya@gmail.com. Tell me a story.", tokenizer, cfg.block_size))
print(check_input("Once upon a time a boy found a knife"))
print(check_output("The dog was stupid and ran away."))

### 🔥 Danger demo: models memorise their training data
What if private data slips into the training set? We hide one "phone number" in 5% of the stories
and fine-tune briefly. Then we simply *ask*.

In [ ]:
SECRET = "Rahul's phone number is 98765 43210"
leaky_stories = []
for s in stories[:3000]:
    if random.random() < 0.05:                                   # poison 5% of the stories
        sentences = s.split(". ")
        sentences.insert(random.randint(0, len(sentences)), SECRET)
        s = ". ".join(sentences)
    leaky_stories.append(s)
leak_data = torch.tensor(tokenizer.encode(EOT.join(leaky_stories)), dtype=torch.long)

leaky_model = copy.deepcopy(model).train()
optimizer = torch.optim.AdamW(leaky_model.parameters(), lr=5e-4)
for step in range(300):
    ix = torch.randint(len(leak_data) - cfg.block_size - 1, (BATCH_SIZE,))
    xb = torch.stack([leak_data[i:i + cfg.block_size] for i in ix]).to(device)
    yb = torch.stack([leak_data[i + 1:i + cfg.block_size + 1] for i in ix]).to(device)
    _, loss = leaky_model(xb, yb)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
leaky_model.eval()

print("Original model :", generate_text(model, tokenizer, "Rahul's phone number is", max_new_tokens=12, temperature=0))
print("Leaky model    :", generate_text(leaky_model, tokenizer, "Rahul's phone number is", max_new_tokens=12, temperature=0))

The model **leaks the secret word-for-word**. This really happens with big LLMs (researchers have extracted
phone numbers and emails from GPT-2's training data). Lessons:
* clean PII out of training data **before** training (India's **DPDP Act 2023** makes this a legal duty)
* check outputs for copied training data (our `MemorizationChecker`):

In [ ]:
checker = MemorizationChecker(text)
story = generate_text(model, tokenizer, "Once upon a time", max_new_tokens=80, temperature=0.8, top_k=40)
print(story, "\n")
print(checker.check(story[len("Once upon a time"):]))

### ⚖️ Bias: the model copies the patterns in its data
First look at the **training data**: after each job word, which pronoun comes first, "he" or "she"?

In [ ]:
def pronoun_after(job):
    counts = Counter()
    for i, w in enumerate(words):
        if w.strip('.,!?"').lower() == job:
            for nxt in words[i + 1:i + 12]:
                n = nxt.strip('.,!?"').lower()
                if n in ("he", "she"):
                    counts[n] += 1
                    break
    return counts

for job in ["doctor", "nurse", "pilot", "teacher", "farmer", "cook"]:
    c = pronoun_after(job)
    print(f"{job:8s} in the data:  he {c['he']:3d}   she {c['she']:3d}")

Now ask **the model**. Does it lean the same way?

In [ ]:
for job in ["doctor", "nurse", "pilot", "teacher", "farmer", "cook"]:
    probs = dict(next_token_table(model, tokenizer, f"The {job} said that", k=200))
    he, she = probs.get(" he", 0), probs.get(" she", 0)
    print(f"The {job:8s} said that ...   P(he) = {he:.3f}   P(she) = {she:.3f}")

Nobody programmed these preferences: they come from the statistics of the training stories.
Real LLMs trained on the internet absorb real-world stereotypes the same way.

**Other risks to know** (OWASP Top 10 for LLM Applications):
* **Prompt injection** (#1): hidden text like *"ignore previous instructions and…"* inside a document or website hijacks the model
* **Jailbreaks**: tricking the model around its safety training ("pretend you are…")
* **Over-reliance**: trusting output without checking. Always keep a human in the loop for important decisions
* **Misuse**: deepfakes, spam, phishing at scale

---
# Part 11 · 📱 Build the app: StoryGPT Studio

Everything we built today comes together in one web app, running **our own models**:
tokenizer ✔ embeddings ✔ transformer ✔ sampling knobs ✔ instruction tuning ✔ hallucination radar ✔ guardrails ✔

In [ ]:
import gradio as gr

MODELS = {"Instruction-tuned (SFT)": sft_model, "Base (pre-trained only)": model}

def confidence_label(p):
    return "sure (>50%)" if p >= 0.5 else "unsure (15-50%)" if p >= 0.15 else "guessing (<15%)"

def studio(request, model_name, temperature, top_k, top_p, max_tokens, guardrails_on):
    report = []
    if guardrails_on:
        request, blocked, messages = check_input(request, tokenizer, cfg.block_size)
        report += messages
        if blocked:
            return [], "\n\n".join(report)
    chosen = MODELS[model_name]
    prompt = chat_format(request) if chosen is sft_model else request     # the chat template, in action
    steps = generate(chosen, tokenizer, prompt, max_new_tokens=int(max_tokens), temperature=temperature,
                     top_k=int(top_k) or None, top_p=top_p)
    story = "".join(t for t, _ in steps)
    highlighted = [(t, confidence_label(p)) for t, p in steps]
    if guardrails_on:
        story, messages = check_output(story)
        report += messages
        if messages:
            highlighted = [(story, "filtered")]
    shaky = low_confidence_tokens(steps)
    report.append(f"🎯 Model confidence: {np.mean([p for _, p in steps]):.0%} average, {len(shaky)} shaky tokens")
    mem = checker.check(story)
    report.append(f"📚 Copied from training data: {mem['copied_fraction']:.0%}")
    report.append(f"🔤 Prompt the model actually saw ({len(tokenizer.encode(prompt))} tokens): `{prompt!r}`")
    return highlighted, "\n\n".join(report)

def tokenize_view(t):
    pieces = tokenizer.tokens(t)
    return ([(p, "A" if i % 2 == 0 else "B") for i, p in enumerate(pieces)],
            f"Our tokenizer: **{len(pieces)}** tokens · GPT-4o tokenizer: **{len(gpt4o.encode(t, disallowed_special=()))}** tokens")

with gr.Blocks(title="StoryGPT Studio") as demo:
    gr.Markdown("# 📖 StoryGPT Studio\nA GenAI app running **our own GPT, trained from scratch today**. No API key.")
    with gr.Tab("✍️ Story Studio"):
        with gr.Row():
            with gr.Column():
                req = gr.Textbox(label="Your request / prompt", value="Write a story about a cat named Rahul.")
                which = gr.Radio(list(MODELS), value="Instruction-tuned (SFT)", label="Model")
                temp = gr.Slider(0, 2, 0.8, step=0.05, label="Temperature")
                topk = gr.Slider(0, 200, 40, step=1, label="Top-k (0 = off)")
                topp = gr.Slider(0.05, 1, 0.95, step=0.05, label="Top-p")
                maxt = gr.Slider(10, 300, 120, step=10, label="Max new tokens")
                guard = gr.Checkbox(True, label="🛡️ Guardrails ON")
                btn = gr.Button("Generate", variant="primary")
            with gr.Column():
                out = gr.HighlightedText(label="Output: coloured by model confidence (hallucination radar)",
                                         color_map={"sure (>50%)": "#22c55e", "unsure (15-50%)": "#facc15",
                                                    "guessing (<15%)": "#ef4444", "filtered": "#9ca3af"},
                                         combine_adjacent=False)
                rep = gr.Markdown()
        btn.click(studio, [req, which, temp, topk, topp, maxt, guard], [out, rep])
        gr.Examples([["Write a story about a dog named Max."], ["Write a story about a bird named Priya."],
                     ["Write a story about an owl named Kovira."],
                     ["My phone is 9876543210. Write a story about a frog named Arjun."],
                     ["Write a story about a boy with a gun."]], inputs=[req])
    with gr.Tab("🔤 Tokenizer"):
        tin = gr.Textbox(label="Text", value="Mujhe AI seekhna pasand hai!")
        tbtn = gr.Button("Tokenize")
        tout = gr.HighlightedText(color_map={"A": "#93c5fd", "B": "#fcd34d"}, show_legend=False,
                                  combine_adjacent=False)
        tstat = gr.Markdown()
        tbtn.click(tokenize_view, tin, [tout, tstat])

demo.launch(share=True, debug=False)   # share=True gives a public link you can open on your phone

---
# Part 12 · Recap: we built a mini ChatGPT pipeline

| Step | What we built | What the big players do |
|---|---|---|
| 📚 Data | 6,000 cleaned stories | trillions of tokens of web, books, code; heavy cleaning & filtering |
| 🔤 Tokenizer | BPE from scratch, 1,079 tokens | byte-level BPE, 100k–200k tokens |
| 🧭 Embeddings | 128-number vectors, learned | thousands of numbers per token |
| 👀 Transformer | 4 layers × 4 heads, ~1M params | ~100 layers, billions of params |
| 🏋️ Pre-training | minutes on 1 GPU | months on thousands of GPUs |
| 🎲 Generation | temperature, top-k, top-p | exactly the same knobs |
| 🎓 Instruction tuning | 800 steps on auto-made examples | ~100k human-written examples + RLHF |
| 🛡️ Safety | regex guardrails, memorization & bias checks | classifiers, red-teaming, policies, audits |
| 📱 App | Gradio | web/mobile apps, APIs, monitoring |

### 🏠 Homework
1. Train with `num_merges=500` and `2000`. How do vocab size, tokens-per-character and loss change?
2. Change `n_layer` / `n_embd`. Plot parameters vs final validation loss.
3. Add your own guardrail (e.g. block a topic, detect a college roll number with a regex).
4. Make a new instruction type for Part 8, e.g. *"Write a sad story about…"*, and fine-tune.

### 🔜 Day 8: RAG
Our model makes up facts because it has **no documents to read**. Tomorrow we use **embeddings** to search
real documents and hand the right paragraphs to the model: **Retrieval-Augmented Generation**,
the #1 way industry reduces hallucination.